# 07 — Export the Power BI Dashboard's Data Workbook

Assembles the flat, denormalised tables the Power BI dashboard needs, from artifacts earlier notebooks/scripts already produce — `df_development.parquet`, the predictions saved by `predict.py` and `06_visualize_predictions.py`, the persisted production CatBoost model + scaler, and `src/clustering.py`'s descriptive clustering functions. No new model is fit and no new result is computed here beyond that assembly — see `src/export.py`'s module docstring.

Output: `outputs/powerbi/TFM_PowerBI_data.xlsx`, ready for Power BI's "Get Data > Excel".

Requires `predict.py` and `06_visualize_predictions.py` to have run first (for the predictions this reads).

In [2]:
import sys

sys.path.append("..")

import pandas as pd
from sklearn.preprocessing import RobustScaler

from src import (
    ID_COLS,
    TARGET,
    EU_REGIONS,
    build_predictor_list,
    aggregate_country_features,
    run_kmeans,
    compute_pca_coords,
    load_artifact,
    build_suicide_rate_panel_table,
    build_predictions_table,
    build_shap_importance_table,
    build_model_comparison_table,
    build_cluster_lookup_table,
    write_powerbi_workbook,
)
from src.explainability import compute_shap_values

N_CLUSTERS = 4  # mirrors EU_REGIONS' 4 groups — see 04_clustering.ipynb

df = pd.read_parquet("../data/processed/df_development.parquet")
predictor_features = build_predictor_list(df, ID_COLS, TARGET)
print(f"df_development: {len(df)} rows | {len(predictor_features)} predictors")

df_development: 594 rows | 18 predictors


## Cluster + PCA lookup

Same descriptive clustering as `04_clustering.ipynb` (k=4 K-Means on the country-level aggregated, scaled features) — recomputed here rather than imported from a saved table, since `04_clustering.py` doesn't currently persist the fitted labels, only the ARI/NMI summary. Feeds both the panel table's Region/Cluster columns and the dashboard's cluster scatter page directly.

In [3]:
agg = aggregate_country_features(df, predictor_features, target=TARGET)
feature_cols = [c for c in agg.columns if c != "Code"]
X_scaled = pd.DataFrame(
    RobustScaler().fit_transform(agg[feature_cols]), columns=feature_cols
)

cluster_labels, _ = run_kmeans(X_scaled, k=N_CLUSTERS)
pca_coords, var_explained = compute_pca_coords(X_scaled)
print(f"PCA explained variance (2 components): {100 * var_explained.sum():.1f}%")

cluster_lookup = build_cluster_lookup_table(
    agg["Code"], cluster_labels, pca_coords, EU_REGIONS
)
cluster_lookup.head()

PCA explained variance (2 components): 48.9%


,Code,Cluster,PCA_Component_1,PCA_Component_2,Region
0,AUT,Cluster 3,-0.311946,0.825639,Western Europe/Nordics
1,BEL,Cluster 3,-0.021782,0.841032,Western Europe/Nordics
2,BGR,Cluster 1,-1.112949,-1.511808,Eastern Europe
3,CYP,Cluster 3,-1.105033,1.979244,Mediterranean
4,CZE,Cluster 1,-0.829355,-0.241916,Eastern Europe


## SHAP importance

Reuses the persisted production CatBoost model and scaler — the exact model behind `predict.py`, not a freshly refit one — so the dashboard's importance ranking always matches whatever model is actually in production.

In [4]:
model = load_artifact("../outputs/models/catboost_option_b.joblib")
scaler = load_artifact("../outputs/models/scaler_option_b.joblib")

X_all = pd.DataFrame(
    scaler.transform(df[predictor_features]), columns=predictor_features
)
_, shap_values, _ = compute_shap_values(model, X_all, sample_size=500, random_state=42)

shap_table = build_shap_importance_table(shap_values, predictor_features)
shap_table.head()

,Feature,Mean_Abs_SHAP
0,Alcohol use disorders,3.084469
1,Bipolar disorder,0.914036
2,Schizophrenia,0.631993
3,Population,0.611510
4,Attention-deficit/hyperactivity disorder,0.571092


## Predictions, model comparison, and the panel table

All three assembled from tables earlier stages already saved to `outputs/tables/` — nothing recomputed.

In [5]:
catboost_predictions = pd.read_parquet("../outputs/tables/predictions.parquet")
sarimax_predictions = pd.read_parquet("../outputs/tables/predictions_temporal.parquet")
predictions_table = build_predictions_table(
    catboost_predictions, sarimax_predictions, cluster_lookup
)

result_tables = {
    ("Option A", "Test"): pd.read_parquet(
        "../outputs/tables/test_geographical.parquet"
    ),
    ("Option A", "Validation"): pd.read_parquet(
        "../outputs/tables/val_geographical.parquet"
    ),
    ("Option B", "Test"): pd.read_parquet("../outputs/tables/test_temporal.parquet"),
    ("Option B", "Validation"): pd.read_parquet(
        "../outputs/tables/val_temporal.parquet"
    ),
}
model_comparison_table = build_model_comparison_table(result_tables)

panel_table = build_suicide_rate_panel_table(
    df, predictor_features, cluster_lookup, target=TARGET
)

temporal_persistence_table = pd.read_parquet(
    "../outputs/tables/temporal_persistence_check.parquet"
)
cluster_agreement_table = pd.read_parquet(
    "../outputs/tables/cluster_region_agreement.parquet"
)

print(
    "panel:",
    panel_table.shape,
    "| predictions:",
    predictions_table.shape,
    "| model comparison:",
    model_comparison_table.shape,
)

panel: (594, 24) | predictions: (108, 6) | model comparison: (24, 8)


## Write the workbook

In [6]:
import os

os.makedirs("../outputs/powerbi", exist_ok=True)

tables = {
    "Suicide_Rate_Panel": panel_table,
    "Predictions_2022_2023": predictions_table,
    "SHAP_Importance": shap_table,
    "Model_Comparison": model_comparison_table,
    "Cluster_PCA": cluster_lookup,
    "Temporal_Persistence_Check": temporal_persistence_table,
    "Cluster_Region_Agreement": cluster_agreement_table,
}

path = write_powerbi_workbook(tables, "../outputs/powerbi/PowerBI_data.xlsx")
print(f"Saved: {path} ({len(tables)} sheets)")

Saved: ../outputs/powerbi/PowerBI_data.xlsx (7 sheets)
